# ARX Algorithm-Only Notebook

Notebook nay chi chay phan thuat toan ARX:
1. Nap/sinh du lieu
2. Chia train/validation/test theo thu tu thoi gian
3. Tao regression matrix
4. Uoc luong OLS
5. Danh gia 1-step, 12-step, free-run
6. Residual diagnostics
7. Model structure search
8. Luu artifact JSON

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from arx_pipeline import (
    DataConfig,
    SplitConfig,
    ModelConfig,
    load_or_generate_data,
    split_time_series,
    build_regression_matrix,
    estimate_ols,
    build_true_theta,
    summarize_parameters,
    compute_ar_roots,
    evaluate_slice,
    model_selection_search,
    evaluate_candidate_order,
    artifact_payload,
)

DATA_CONFIG = DataConfig(
    csv_path=Path("greenhouse_data.csv"),
    generator_script_path=Path("data_generator.py"),
    force_regenerate_from_script=False,
    auto_save_generated_csv=True,
    generated_days=365,
    generated_sampling_seconds=300,
    generated_seed=42,
    generated_start_date="2025-01-01",
)

SPLIT_CONFIG = SplitConfig(train_ratio=0.60, val_ratio=0.20)
MODEL_CONFIG = ModelConfig(na=2, nb=2, nk=1, include_intercept=False)

In [2]:
df_full, true_params, data_source = load_or_generate_data(DATA_CONFIG)
df_train, df_val, df_test = split_time_series(df_full, SPLIT_CONFIG)

overview = pd.Series({
    "data_source": data_source,
    "rows_full": len(df_full),
    "rows_train": len(df_train),
    "rows_val": len(df_val),
    "rows_test": len(df_test),
    "timestamp_start": str(df_full["Timestamp"].iloc[0]),
    "timestamp_end": str(df_full["Timestamp"].iloc[-1]),
    "months_present": sorted(int(m) for m in pd.Series(df_full["Month"]).dropna().unique()),
    "seasons_present": sorted(str(s) for s in pd.Series(df_full["Season"]).dropna().unique()),
})

overview

data_source                        CSV:greenhouse_data.csv
rows_full                                           105120
rows_train                                           63072
rows_val                                             21024
rows_test                                            21024
timestamp_start                        2025-01-01 00:00:00
timestamp_end                          2025-12-31 23:55:00
months_present     [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
seasons_present           [autumn, spring, summer, winter]
dtype: object

In [3]:
x_train, y_train = build_regression_matrix(df_train, MODEL_CONFIG)
theta_hat, cov_hat, sigma2_hat = estimate_ols(x_train, y_train)
true_theta = build_true_theta(true_params, MODEL_CONFIG)

train_matrix_info = pd.Series({
    "x_train_shape": x_train.shape,
    "y_train_shape": y_train.shape,
    "rank_x_train": int(np.linalg.matrix_rank(x_train)),
    "n_params": len(MODEL_CONFIG.param_names),
    "cond_xtx": float(np.linalg.cond(x_train.T @ x_train)),
    "sigma2_hat": float(sigma2_hat),
})

train_matrix_info

x_train_shape        (63070, 14)
y_train_shape           (63070,)
rank_x_train                  14
n_params                      14
cond_xtx         56303331.052535
sigma2_hat               0.06248
dtype: object

In [4]:
params_df = pd.DataFrame(
    summarize_parameters(theta_hat, cov_hat, MODEL_CONFIG, true_params)
)
roots_df = pd.DataFrame(compute_ar_roots(theta_hat, MODEL_CONFIG))

params_display = params_df[[
    "name",
    "estimate",
    "std",
    "ci95_low",
    "ci95_high",
    "true_value",
    "delta_vs_true",
    "sign_ok",
]].copy()

sign_ok_count = int(params_df["sign_ok"].sum())
sign_total = int(len(params_df))
print(f"Sign recovery: {sign_ok_count}/{sign_total}")

params_display.round(6), roots_df.round(6)

Sign recovery: 14/14


(               name  estimate       std  ci95_low  ci95_high  true_value  \
 0                a1  0.963098  0.001670  0.959824   0.966371     0.96500   
 1                a2  0.026738  0.001724  0.023358   0.030118     0.02500   
 2   b_Temperature_1 -0.007410  0.002548 -0.012403  -0.002416    -0.00800   
 3   b_Temperature_2 -0.003922  0.002493 -0.008808   0.000964    -0.00400   
 4      b_Humidity_1  0.002567  0.000836  0.000929   0.004206     0.00250   
 5      b_Humidity_2  0.001029  0.000825 -0.000588   0.002646     0.00120   
 6         b_Light_1 -0.000132  0.000070 -0.000270   0.000005    -0.00022   
 7         b_Light_2 -0.000194  0.000071 -0.000334  -0.000055    -0.00010   
 8          b_Drip_1  1.251167  0.003388  1.244526   1.257809     1.25000   
 9          b_Drip_2  1.851585  0.005452  1.840899   1.862271     1.85000   
 10         b_Mist_1  0.038049  0.010848  0.016787   0.059311     0.05000   
 11         b_Mist_2  0.035212  0.005013  0.025385   0.045038     0.03000   

In [5]:
train_eval = evaluate_slice("Train", df_train, theta_hat, MODEL_CONFIG, true_theta=true_theta, n_step=12)
val_eval = evaluate_slice("Validation", df_val, theta_hat, MODEL_CONFIG, true_theta=true_theta, n_step=12)
test_eval = evaluate_slice("Test", df_test, theta_hat, MODEL_CONFIG, true_theta=true_theta, n_step=12)

metrics_table = pd.DataFrame([
    {
        "Split": "Train",
        "FIT_1step": train_eval["metrics_1step"]["FIT"],
        "FIT_12step": train_eval["metrics_n_step"]["FIT"],
        "FIT_sim": train_eval["metrics_sim"]["FIT"],
        "Theo_FIT_sim": train_eval.get("theoretical_max_free_run", {}).get("FIT", np.nan),
        "RMSE_1step": train_eval["metrics_1step"]["RMSE"],
        "RMSE_sim": train_eval["metrics_sim"]["RMSE"],
    },
    {
        "Split": "Validation",
        "FIT_1step": val_eval["metrics_1step"]["FIT"],
        "FIT_12step": val_eval["metrics_n_step"]["FIT"],
        "FIT_sim": val_eval["metrics_sim"]["FIT"],
        "Theo_FIT_sim": val_eval.get("theoretical_max_free_run", {}).get("FIT", np.nan),
        "RMSE_1step": val_eval["metrics_1step"]["RMSE"],
        "RMSE_sim": val_eval["metrics_sim"]["RMSE"],
    },
    {
        "Split": "Test",
        "FIT_1step": test_eval["metrics_1step"]["FIT"],
        "FIT_12step": test_eval["metrics_n_step"]["FIT"],
        "FIT_sim": test_eval["metrics_sim"]["FIT"],
        "Theo_FIT_sim": test_eval.get("theoretical_max_free_run", {}).get("FIT", np.nan),
        "RMSE_1step": test_eval["metrics_1step"]["RMSE"],
        "RMSE_sim": test_eval["metrics_sim"]["RMSE"],
    },
])

metrics_table.round(4)

,Split,FIT_1step,FIT_12step,FIT_sim,Theo_FIT_sim,RMSE_1step,RMSE_sim
0,Train,92.5471,76.4447,49.5463,49.0595,0.2499,1.6920
1,Validation,91.6406,73.1094,42.9588,42.2497,0.2510,1.7124
2,Test,91.3491,72.6736,43.8749,43.2691,0.2520,1.6347


In [6]:
diag_table = pd.DataFrame([
    {
        "Split": "Validation",
        "mean": val_eval["residual_diagnostics"]["mean"],
        "std": val_eval["residual_diagnostics"]["std"],
        "shapiro_p": val_eval["residual_diagnostics"]["normality"]["shapiro_pvalue"],
        "dagostino_p": val_eval["residual_diagnostics"]["normality"]["dagostino_pvalue"],
        "ljung_box_pass": val_eval["residual_diagnostics"]["ljung_box"]["passes_all_lags"],
        "failed_lags": val_eval["residual_diagnostics"]["ljung_box"]["failed_lags"],
    },
    {
        "Split": "Test",
        "mean": test_eval["residual_diagnostics"]["mean"],
        "std": test_eval["residual_diagnostics"]["std"],
        "shapiro_p": test_eval["residual_diagnostics"]["normality"]["shapiro_pvalue"],
        "dagostino_p": test_eval["residual_diagnostics"]["normality"]["dagostino_pvalue"],
        "ljung_box_pass": test_eval["residual_diagnostics"]["ljung_box"]["passes_all_lags"],
        "failed_lags": test_eval["residual_diagnostics"]["ljung_box"]["failed_lags"],
    },
])

diag_table.round(4)

,Split,mean,std,shapiro_p,dagostino_p,ljung_box_pass,failed_lags
0,Validation,0.0016,0.251,0.2024,0.4118,True,[]
1,Test,0.0005,0.252,0.7259,0.1141,True,[]


In [7]:
selection_df = model_selection_search(
    df_train=df_train,
    df_val=df_val,
    base_model_config=MODEL_CONFIG,
    na_list=[1, 2, 3],
    nb_list=[1, 2, 3],
    nk_list=[1, 2],
)

selection_df.head(10).round(4)

,na,nb,nk,n_params,RMSE_1step,FIT_1step,R2_1step,AIC_1step,BIC_1step,RMSE_sim,FIT_sim,R2_sim
0,3,1,1,9,0.4102,86.3359,0.9813,-37443.8834,-37372.3039,1.5035,49.9185,0.7492
1,2,1,1,8,0.4209,85.9812,0.9803,-36371.2618,-36307.6352,1.6094,46.3905,0.7126
2,1,2,1,13,0.2514,91.6262,0.9930,-58026.2945,-57922.9013,1.6211,46.0003,0.7084
3,3,1,2,9,0.4516,84.9589,0.9774,-33407.2784,-33335.6989,1.7004,43.3621,0.6792
4,1,3,1,19,0.2511,91.6375,0.9930,-58066.9289,-57915.8166,1.7079,43.1125,0.6764
5,3,2,1,15,0.2509,91.6417,0.9930,-58096.4441,-57977.1449,1.7116,42.9892,0.6750
6,2,2,1,14,0.2510,91.6406,0.9930,-58096.5147,-57985.1682,1.7124,42.9588,0.6746
7,2,3,1,20,0.2510,91.6406,0.9930,-58080.5684,-57921.5029,1.7201,42.7066,0.6717
8,3,3,1,21,0.2510,91.6406,0.9930,-58078.5459,-57911.5271,1.7216,42.6546,0.6712
9,2,1,2,8,0.4542,84.8712,0.9771,-33167.4943,-33103.8677,1.7320,42.3082,0.6672


In [8]:
if selection_df.empty:
    raise RuntimeError("Model selection rong, kiem tra lai du lieu/config")

best_row = selection_df.iloc[0]
best_candidate = evaluate_candidate_order(
    df_train=df_train,
    df_val=df_val,
    df_test=df_test,
    base_model_config=MODEL_CONFIG,
    na=int(best_row["na"]),
    nb=int(best_row["nb"]),
    nk=int(best_row["nk"]),
)

best_cfg = best_candidate["model_config"]
comparison_df = pd.DataFrame([
    {
        "Model": f"Baseline ARX({MODEL_CONFIG.na},{MODEL_CONFIG.nb},{MODEL_CONFIG.nk})",
        "Val_FIT_sim": val_eval["metrics_sim"]["FIT"],
        "Test_FIT_sim": test_eval["metrics_sim"]["FIT"],
        "Val_RMSE_sim": val_eval["metrics_sim"]["RMSE"],
        "Test_RMSE_sim": test_eval["metrics_sim"]["RMSE"],
    },
    {
        "Model": f"Best ARX({best_cfg.na},{best_cfg.nb},{best_cfg.nk})",
        "Val_FIT_sim": best_candidate["val"]["metrics_sim"]["FIT"],
        "Test_FIT_sim": best_candidate["test"]["metrics_sim"]["FIT"],
        "Val_RMSE_sim": best_candidate["val"]["metrics_sim"]["RMSE"],
        "Test_RMSE_sim": best_candidate["test"]["metrics_sim"]["RMSE"],
    },
])

comparison_df.round(4)

,Model,Val_FIT_sim,Test_FIT_sim,Val_RMSE_sim,Test_RMSE_sim
0,"Baseline ARX(2,2,1)",42.9588,43.8749,1.7124,1.6347
1,"Best ARX(3,1,1)",49.9185,48.1578,1.5035,1.5100


In [9]:
results_algo = {
    "data_source": data_source,
    "data_config": DATA_CONFIG,
    "split_config": SPLIT_CONFIG,
    "model_config": MODEL_CONFIG,
    "df_full": df_full,
    "df_train": df_train,
    "df_val": df_val,
    "df_test": df_test,
    "true_params": true_params,
    "dataset_overview": {
        "rows": int(len(df_full)),
        "timestamp_start": str(df_full["Timestamp"].iloc[0]),
        "timestamp_end": str(df_full["Timestamp"].iloc[-1]),
        "months_present": sorted(int(m) for m in pd.Series(df_full["Month"]).dropna().unique()),
        "seasons_present": sorted(str(s) for s in pd.Series(df_full["Season"]).dropna().unique()),
        "condition_number_xtx": float(np.linalg.cond(x_train.T @ x_train)),
        "rank_x_train": int(np.linalg.matrix_rank(x_train)),
        "n_params": int(len(MODEL_CONFIG.param_names)),
    },
    "theta_hat": theta_hat.tolist(),
    "sigma2": float(sigma2_hat),
    "ar_roots": roots_df.to_dict(orient="records"),
    "parameter_summary": params_df.to_dict(orient="records"),
    "train": train_eval,
    "val": val_eval,
    "test": test_eval,
    "model_selection": selection_df,
    "best_candidate": best_candidate,
}

payload = artifact_payload(results_algo)
output_path = Path("arx_model_algo_only.json")
with output_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)
    f.write("\n")

print(f"Saved: {output_path}")

Saved: arx_model_algo_only.json


In [10]:
baseline_val_fit = float(val_eval["metrics_sim"]["FIT"])
baseline_test_fit = float(test_eval["metrics_sim"]["FIT"])
best_val_fit = float(best_candidate["val"]["metrics_sim"]["FIT"])
best_test_fit = float(best_candidate["test"]["metrics_sim"]["FIT"])

print("BASELINE FIT_sim:", {"val": round(baseline_val_fit, 3), "test": round(baseline_test_fit, 3)})
print("BEST_CANDIDATE FIT_sim:", {"val": round(best_val_fit, 3), "test": round(best_test_fit, 3)})
print("GAIN:", {"val": round(best_val_fit - baseline_val_fit, 3), "test": round(best_test_fit - baseline_test_fit, 3)})

BASELINE FIT_sim: {'val': 42.959, 'test': 43.875}
BEST_CANDIDATE FIT_sim: {'val': 49.919, 'test': 48.158}
GAIN: {'val': 6.96, 'test': 4.283}


In [11]:
print(df_full.columns.tolist())

['Timestamp', 'Month', 'Season', 'Soil_Moisture', 'Soil_Low_SP', 'Soil_High_SP', 'Temperature', 'Humidity', 'Light', 'Drip', 'Mist', 'Fan']


In [12]:
def build_augmented_df(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    df["Light_log"] = np.log1p(df["Light"].clip(lower=0))
    df["Temp_x_Humi"] = df["Temperature"] * df["Humidity"]
    df["Temp_x_Light"] = df["Temperature"] * df["Light_log"]
    df["Humi_x_Light"] = df["Humidity"] * df["Light_log"]
    df["SP_Center"] = 0.5 * (df["Soil_Low_SP"] + df["Soil_High_SP"])
    df["SP_Width"] = (df["Soil_High_SP"] - df["Soil_Low_SP"])
    df["Month_sin"] = np.sin(2.0 * np.pi * df["Month"] / 12.0)
    df["Month_cos"] = np.cos(2.0 * np.pi * df["Month"] / 12.0)

    season_map = {
        "Spring": 0,
        "Summer": 1,
        "Autumn": 2,
        "Winter": 3,
    }
    season_num = df["Season"].map(season_map).fillna(0).astype(int)
    df["Season_sin"] = np.sin(2.0 * np.pi * season_num / 4.0)
    df["Season_cos"] = np.cos(2.0 * np.pi * season_num / 4.0)
    return df


def run_scenario(
    scenario_name: str,
    df_source: pd.DataFrame,
    input_cols: tuple[str, ...],
    na_list: list[int],
    nb_list: list[int],
    nk_list: list[int],
    include_intercept: bool,
    simulation_clip: tuple[float, float] | None,
):
    df_train_s, df_val_s, df_test_s = split_time_series(df_source, SPLIT_CONFIG)

    base_cfg = ModelConfig(
        na=2,
        nb=2,
        nk=1,
        include_intercept=include_intercept,
        input_cols=input_cols,
        output_col="Soil_Moisture",
        simulation_clip=simulation_clip,
    )

    search_df = model_selection_search(
        df_train=df_train_s,
        df_val=df_val_s,
        base_model_config=base_cfg,
        na_list=na_list,
        nb_list=nb_list,
        nk_list=nk_list,
    )

    if search_df.empty:
        raise RuntimeError(f"{scenario_name}: model search returned empty")

    best = search_df.iloc[0]
    best_eval = evaluate_candidate_order(
        df_train=df_train_s,
        df_val=df_val_s,
        df_test=df_test_s,
        base_model_config=base_cfg,
        na=int(best["na"]),
        nb=int(best["nb"]),
        nk=int(best["nk"]),
    )

    return {
        "scenario": scenario_name,
        "input_dim": len(input_cols),
        "best_cfg": (int(best["na"]), int(best["nb"]), int(best["nk"])),
        "val_fit_sim": float(best_eval["val"]["metrics_sim"]["FIT"]),
        "test_fit_sim": float(best_eval["test"]["metrics_sim"]["FIT"]),
        "val_rmse_sim": float(best_eval["val"]["metrics_sim"]["RMSE"]),
        "test_rmse_sim": float(best_eval["test"]["metrics_sim"]["RMSE"]),
        "raw": best_eval,
    }


baseline_result = {
    "scenario": "baseline_current",
    "input_dim": len(MODEL_CONFIG.input_cols),
    "best_cfg": (MODEL_CONFIG.na, MODEL_CONFIG.nb, MODEL_CONFIG.nk),
    "val_fit_sim": float(val_eval["metrics_sim"]["FIT"]),
    "test_fit_sim": float(test_eval["metrics_sim"]["FIT"]),
    "val_rmse_sim": float(val_eval["metrics_sim"]["RMSE"]),
    "test_rmse_sim": float(test_eval["metrics_sim"]["RMSE"]),
    "raw": None,
}


df_aug = build_augmented_df(df_full)

# Clip theo phan vi train de giam drift free-run
_df_train_for_clip, _, _ = split_time_series(df_aug, SPLIT_CONFIG)
clip_low = float(_df_train_for_clip["Soil_Moisture"].quantile(0.01))
clip_high = float(_df_train_for_clip["Soil_Moisture"].quantile(0.99))

scenarios = []
scenarios.append(baseline_result)

# 1) Mo rong order tren input goc
scenarios.append(
    run_scenario(
        scenario_name="wide_order_raw_inputs",
        df_source=df_full,
        input_cols=tuple(["Temperature", "Humidity", "Light", "Drip", "Mist", "Fan"]),
        na_list=[2, 3, 4, 5, 6, 7],
        nb_list=[2, 3, 4, 5],
        nk_list=[1, 2, 3],
        include_intercept=True,
        simulation_clip=None,
    )
)

# 2) Feature engineering + order search
aug_input_cols = tuple([
    "Temperature", "Humidity", "Light", "Drip", "Mist", "Fan",
    "Light_log", "Temp_x_Humi", "Temp_x_Light", "Humi_x_Light",
    "SP_Center", "SP_Width", "Month_sin", "Month_cos", "Season_sin", "Season_cos",
])
scenarios.append(
    run_scenario(
        scenario_name="augmented_features",
        df_source=df_aug,
        input_cols=aug_input_cols,
        na_list=[2, 3, 4, 5, 6],
        nb_list=[1, 2, 3, 4],
        nk_list=[1, 2],
        include_intercept=True,
        simulation_clip=None,
    )
)

# 3) Feature engineering + clip free-run drift
scenarios.append(
    run_scenario(
        scenario_name="augmented_features_with_clip",
        df_source=df_aug,
        input_cols=aug_input_cols,
        na_list=[2, 3, 4, 5, 6],
        nb_list=[1, 2, 3, 4],
        nk_list=[1, 2],
        include_intercept=True,
        simulation_clip=(clip_low, clip_high),
    )
)

scenario_df = pd.DataFrame([{k: v for k, v in s.items() if k != "raw"} for s in scenarios])
scenario_df = scenario_df.sort_values(["test_fit_sim", "val_fit_sim"], ascending=False).reset_index(drop=True)
scenario_df

c:\Users\minht\OneDrive\Desktop\ARX-Model-demoadada\ARX-Model-demov3\arx_pipeline.py:276: RuntimeWarning: overflow encountered in dot
  y_next = float(np.dot(row, theta))
c:\Users\minht\OneDrive\Desktop\ARX-Model-demoadada\ARX-Model-demov3\arx_pipeline.py:276: RuntimeWarning: invalid value encountered in dot
  y_next = float(np.dot(row, theta))
c:\Users\minht\OneDrive\Desktop\ARX-Model-demoadada\ARX-Model-demov3\arx_pipeline.py:276: RuntimeWarning: overflow encountered in dot
  y_next = float(np.dot(row, theta))
c:\Users\minht\OneDrive\Desktop\ARX-Model-demoadada\ARX-Model-demov3\arx_pipeline.py:276: RuntimeWarning: invalid value encountered in dot
  y_next = float(np.dot(row, theta))
c:\Users\minht\OneDrive\Desktop\ARX-Model-demoadada\ARX-Model-demov3\arx_pipeline.py:276: RuntimeWarning: overflow encountered in dot
  y_next = float(np.dot(row, theta))
c:\Users\minht\OneDrive\Desktop\ARX-Model-demoadada\ARX-Model-demov3\arx_pipeline.py:276: RuntimeWarning: invalid value encountered in 

,scenario,input_dim,best_cfg,val_fit_sim,test_fit_sim,val_rmse_sim,test_rmse_sim
0,augmented_features_with_clip,16,"(5, 1, 2)",68.860865,66.414352,0.934891,0.978307
1,augmented_features,16,"(5, 1, 2)",69.336626,66.316387,0.920607,0.981160
2,baseline_current,6,"(2, 2, 1)",42.958845,43.874948,1.712439,1.634739
3,wide_order_raw_inputs,6,"(7, 2, 2)",53.990297,31.342195,1.381377,2.000001


In [13]:
improved_row = scenario_df.iloc[0]
improved_scenario = improved_row["scenario"]
improved_entry = next(s for s in scenarios if s["scenario"] == improved_scenario)
improved_result = improved_entry["raw"]

if improved_result is None:
    raise RuntimeError("Improved scenario is invalid")

baseline_test = float(test_eval["metrics_sim"]["FIT"])
improved_test = float(improved_result["test"]["metrics_sim"]["FIT"])
baseline_val = float(val_eval["metrics_sim"]["FIT"])
improved_val = float(improved_result["val"]["metrics_sim"]["FIT"])

improved_summary = pd.DataFrame([
    {
        "metric": "FIT_sim (Validation)",
        "baseline": baseline_val,
        "improved": improved_val,
        "gain": improved_val - baseline_val,
    },
    {
        "metric": "FIT_sim (Test)",
        "baseline": baseline_test,
        "improved": improved_test,
        "gain": improved_test - baseline_test,
    },
])

print("Improved scenario:", improved_scenario)
print("Improved ARX config:", improved_result["model_config"])
improved_summary.round(3)

Improved scenario: augmented_features_with_clip
Improved ARX config: ModelConfig(na=5, nb=1, nk=2, include_intercept=True, input_cols=('Temperature', 'Humidity', 'Light', 'Drip', 'Mist', 'Fan', 'Light_log', 'Temp_x_Humi', 'Temp_x_Light', 'Humi_x_Light', 'SP_Center', 'SP_Width', 'Month_sin', 'Month_cos', 'Season_sin', 'Season_cos'), output_col='Soil_Moisture', simulation_clip=(50.5465794049447, 64.61272806162447))


,metric,baseline,improved,gain
0,FIT_sim (Validation),42.959,68.861,25.902
1,FIT_sim (Test),43.875,66.414,22.539


In [14]:
# Build full artifact for improved model
if improved_scenario.startswith("augmented"):
    df_for_model = df_aug
else:
    df_for_model = df_full

df_train_imp, df_val_imp, df_test_imp = split_time_series(df_for_model, SPLIT_CONFIG)
improved_cfg = improved_result["model_config"]

x_train_imp, y_train_imp = build_regression_matrix(df_train_imp, improved_cfg)
theta_imp, cov_imp, sigma2_imp = estimate_ols(x_train_imp, y_train_imp)

train_eval_imp = evaluate_slice("Train", df_train_imp, theta_imp, improved_cfg, true_theta=None, n_step=12)
val_eval_imp = evaluate_slice("Validation", df_val_imp, theta_imp, improved_cfg, true_theta=None, n_step=12)
test_eval_imp = evaluate_slice("Test", df_test_imp, theta_imp, improved_cfg, true_theta=None, n_step=12)

params_imp_df = pd.DataFrame(
    summarize_parameters(theta_imp, cov_imp, improved_cfg, true_params)
)
roots_imp_df = pd.DataFrame(compute_ar_roots(theta_imp, improved_cfg))

results_algo_improved = {
    "data_source": f"{data_source}|{improved_scenario}",
    "data_config": DATA_CONFIG,
    "split_config": SPLIT_CONFIG,
    "model_config": improved_cfg,
    "df_full": df_for_model,
    "df_train": df_train_imp,
    "df_val": df_val_imp,
    "df_test": df_test_imp,
    "true_params": true_params,
    "dataset_overview": {
        "rows": int(len(df_for_model)),
        "timestamp_start": str(df_for_model["Timestamp"].iloc[0]),
        "timestamp_end": str(df_for_model["Timestamp"].iloc[-1]),
        "months_present": sorted(int(m) for m in pd.Series(df_for_model["Month"]).dropna().unique()),
        "seasons_present": sorted(str(s) for s in pd.Series(df_for_model["Season"]).dropna().unique()),
        "condition_number_xtx": float(np.linalg.cond(x_train_imp.T @ x_train_imp)),
        "rank_x_train": int(np.linalg.matrix_rank(x_train_imp)),
        "n_params": int(len(improved_cfg.param_names)),
    },
    "theta_hat": theta_imp.tolist(),
    "sigma2": float(sigma2_imp),
    "ar_roots": roots_imp_df.to_dict(orient="records"),
    "parameter_summary": params_imp_df.to_dict(orient="records"),
    "train": train_eval_imp,
    "val": val_eval_imp,
    "test": test_eval_imp,
    "model_selection": selection_df,
    "best_candidate": improved_result,
}

payload_improved = artifact_payload(results_algo_improved)
output_improved = Path("arx_model_algo_improved.json")
with output_improved.open("w", encoding="utf-8") as f:
    json.dump(payload_improved, f, indent=2)
    f.write("\n")

print(f"Saved improved artifact: {output_improved}")
print("Improved FIT_sim:", {
    "val": round(val_eval_imp["metrics_sim"]["FIT"], 3),
    "test": round(test_eval_imp["metrics_sim"]["FIT"], 3),
})

c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-p

Saved improved artifact: arx_model_algo_improved.json
Improved FIT_sim: {'val': 68.861, 'test': 66.414}


c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-p

In [15]:
# Fair ablation for fixed order ARX(2,2,1)
fixed_221_rows = []

# A) raw features, no intercept, no clip (same as baseline)
res_a = run_scenario(
    scenario_name="fixed_221_raw_no_intercept_no_clip",
    df_source=df_full,
    input_cols=tuple(["Temperature", "Humidity", "Light", "Drip", "Mist", "Fan"]),
    na_list=[2], nb_list=[2], nk_list=[1],
    include_intercept=False,
    simulation_clip=None,
)
fixed_221_rows.append(res_a)

# B) raw features, intercept on
res_b = run_scenario(
    scenario_name="fixed_221_raw_intercept",
    df_source=df_full,
    input_cols=tuple(["Temperature", "Humidity", "Light", "Drip", "Mist", "Fan"]),
    na_list=[2], nb_list=[2], nk_list=[1],
    include_intercept=True,
    simulation_clip=None,
)
fixed_221_rows.append(res_b)

# C) augmented features, intercept on
res_c = run_scenario(
    scenario_name="fixed_221_augmented_intercept",
    df_source=df_aug,
    input_cols=aug_input_cols,
    na_list=[2], nb_list=[2], nk_list=[1],
    include_intercept=True,
    simulation_clip=None,
)
fixed_221_rows.append(res_c)

# D) augmented features, intercept on, clip on
res_d = run_scenario(
    scenario_name="fixed_221_augmented_intercept_clip",
    df_source=df_aug,
    input_cols=aug_input_cols,
    na_list=[2], nb_list=[2], nk_list=[1],
    include_intercept=True,
    simulation_clip=(clip_low, clip_high),
)
fixed_221_rows.append(res_d)

fixed_221_df = pd.DataFrame([{k: v for k, v in r.items() if k != "raw"} for r in fixed_221_rows])
fixed_221_df = fixed_221_df.sort_values(["test_fit_sim", "val_fit_sim"], ascending=False).reset_index(drop=True)
fixed_221_df

c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-p

,scenario,input_dim,best_cfg,val_fit_sim,test_fit_sim,val_rmse_sim,test_rmse_sim
0,fixed_221_augmented_intercept_clip,16,"(2, 2, 1)",48.367699,53.448313,1.550060,1.355898
1,fixed_221_raw_no_intercept_no_clip,6,"(2, 2, 1)",42.958845,43.874948,1.712439,1.634739
2,fixed_221_raw_intercept,6,"(2, 2, 1)",42.485979,43.398773,1.726635,1.648609
3,fixed_221_augmented_intercept,16,"(2, 2, 1)",38.215463,40.675831,1.854841,1.727919


In [16]:
# Show prediction FIT for pre1 (1-step), pre12 (12-step), and free-run

def _pred_fit_from_entry(entry: dict, baseline_val: dict | None = None, baseline_test: dict | None = None):
    raw = entry.get("raw", None)
    if raw is None:
        # baseline_current in scenarios uses current notebook baseline
        return {
            "val_fit_1step": float(baseline_val["metrics_1step"]["FIT"]),
            "val_fit_12step": float(baseline_val["metrics_n_step"]["FIT"]),
            "val_fit_sim": float(baseline_val["metrics_sim"]["FIT"]),
            "test_fit_1step": float(baseline_test["metrics_1step"]["FIT"]),
            "test_fit_12step": float(baseline_test["metrics_n_step"]["FIT"]),
            "test_fit_sim": float(baseline_test["metrics_sim"]["FIT"]),
        }

    return {
        "val_fit_1step": float(raw["val"]["metrics_1step"]["FIT"]),
        "val_fit_12step": float(raw["val"]["metrics_n_step"]["FIT"]),
        "val_fit_sim": float(raw["val"]["metrics_sim"]["FIT"]),
        "test_fit_1step": float(raw["test"]["metrics_1step"]["FIT"]),
        "test_fit_12step": float(raw["test"]["metrics_n_step"]["FIT"]),
        "test_fit_sim": float(raw["test"]["metrics_sim"]["FIT"]),
    }


# 1) All scenarios table
scenario_pred_rows = []
for s in scenarios:
    row = {
        "scenario": s["scenario"],
        "input_dim": s["input_dim"],
        "best_cfg": s["best_cfg"],
    }
    row.update(_pred_fit_from_entry(s, baseline_val=val_eval, baseline_test=test_eval))
    scenario_pred_rows.append(row)

scenario_pred_fit_df = pd.DataFrame(scenario_pred_rows).sort_values(
    ["test_fit_sim", "val_fit_sim"], ascending=False
).reset_index(drop=True)

# 2) Fixed ARX(2,2,1) ablation table
fixed_221_pred_rows = []
for s in fixed_221_rows:
    row = {
        "scenario": s["scenario"],
        "input_dim": s["input_dim"],
        "best_cfg": s["best_cfg"],
    }
    row.update(_pred_fit_from_entry(s, baseline_val=val_eval, baseline_test=test_eval))
    fixed_221_pred_rows.append(row)

fixed_221_pred_fit_df = pd.DataFrame(fixed_221_pred_rows).sort_values(
    ["test_fit_sim", "val_fit_sim"], ascending=False
).reset_index(drop=True)

print("=== ALL SCENARIOS: pre1 / pre12 / sim ===")
display(scenario_pred_fit_df.round(3))

print("=== FIXED ARX(2,2,1): pre1 / pre12 / sim ===")
display(fixed_221_pred_fit_df.round(3))

=== ALL SCENARIOS: pre1 / pre12 / sim ===


,scenario,input_dim,best_cfg,val_fit_1step,val_fit_12step,val_fit_sim,test_fit_1step,test_fit_12step,test_fit_sim
0,augmented_features_with_clip,16,"(5, 1, 2)",86.299,69.816,68.861,85.849,67.158,66.414
1,augmented_features,16,"(5, 1, 2)",86.299,70.279,69.337,85.849,67.107,66.316
2,baseline_current,6,"(2, 2, 1)",91.641,73.109,42.959,91.349,72.674,43.875
3,wide_order_raw_inputs,6,"(7, 2, 2)",86.026,66.591,53.990,85.425,58.394,31.342


=== FIXED ARX(2,2,1): pre1 / pre12 / sim ===


,scenario,input_dim,best_cfg,val_fit_1step,val_fit_12step,val_fit_sim,test_fit_1step,test_fit_12step,test_fit_sim
0,fixed_221_augmented_intercept_clip,16,"(2, 2, 1)",91.636,72.463,48.368,91.345,72.617,53.448
1,fixed_221_raw_no_intercept_no_clip,6,"(2, 2, 1)",91.641,73.109,42.959,91.349,72.674,43.875
2,fixed_221_raw_intercept,6,"(2, 2, 1)",91.641,73.092,42.486,91.349,72.659,43.399
3,fixed_221_augmented_intercept,16,"(2, 2, 1)",91.636,72.933,38.215,91.345,72.577,40.676


In [17]:
# Diagnostic: sweep n-step horizon to compare with free-run

def horizon_sweep(df_slice, theta, model_cfg, name: str, horizons=(1,2,4,8,12,24,48,96)):
    rows = []
    # free-run reference
    sim_eval = evaluate_slice(name, df_slice, theta, model_cfg, true_theta=None, n_step=12)
    fit_sim = float(sim_eval["metrics_sim"]["FIT"])

    for h in horizons:
        ev = evaluate_slice(name, df_slice, theta, model_cfg, true_theta=None, n_step=int(h))
        rows.append({
            "model": name,
            "n_step": int(h),
            "FIT_n_step": float(ev["metrics_n_step"]["FIT"]),
            "RMSE_n_step": float(ev["metrics_n_step"]["RMSE"]),
            "FIT_sim_ref": fit_sim,
            "gap_vs_sim": float(ev["metrics_n_step"]["FIT"] - fit_sim),
        })
    return pd.DataFrame(rows)

# Baseline on validation
baseline_theta = theta_hat
baseline_cfg = MODEL_CONFIG
baseline_sweep_val = horizon_sweep(df_val, baseline_theta, baseline_cfg, "baseline_val")

# Improved on validation
improved_sweep_val = horizon_sweep(df_val_imp, theta_imp, improved_cfg, "improved_val")

print("=== Horizon sweep: baseline (validation) ===")
display(baseline_sweep_val.round(3))

print("=== Horizon sweep: improved (validation) ===")
display(improved_sweep_val.round(3))

c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-p

=== Horizon sweep: baseline (validation) ===


c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-p

,model,n_step,FIT_n_step,RMSE_n_step,FIT_sim_ref,gap_vs_sim
0,baseline_val,1,91.641,0.251,42.959,48.682
1,baseline_val,2,88.398,0.348,42.959,45.439
2,baseline_val,4,83.933,0.482,42.959,40.974
3,baseline_val,8,77.678,0.670,42.959,34.719
4,baseline_val,12,73.109,0.807,42.959,30.151
5,baseline_val,24,63.934,1.083,42.959,20.975
6,baseline_val,48,54.208,1.375,42.959,11.249
7,baseline_val,96,47.045,1.590,42.959,4.086


=== Horizon sweep: improved (validation) ===


,model,n_step,FIT_n_step,RMSE_n_step,FIT_sim_ref,gap_vs_sim
0,improved_val,1,86.060,0.419,68.861,17.199
1,improved_val,2,81.436,0.557,68.861,12.576
2,improved_val,4,75.670,0.730,68.861,6.809
3,improved_val,8,71.124,0.867,68.861,2.263
4,improved_val,12,69.816,0.906,68.861,0.955
5,improved_val,24,69.068,0.929,68.861,0.207
6,improved_val,48,68.861,0.935,68.861,-0.000
7,improved_val,96,68.861,0.935,68.861,-0.000


In [18]:
# Evidence table: baseline vs improved across key metrics

def collect_mode_metrics(evaluation: dict, mode_key: str):
    m = evaluation[mode_key]
    return {
        "FIT": float(m["FIT"]),
        "RMSE": float(m["RMSE"]),
        "MAE": float(m["MAE"]),
        "Bias": float(m["Bias"]),
        "R2": float(m["R2"]),
        "AIC": float(m["AIC"]),
        "BIC": float(m["BIC"]),
    }


def compare_eval(split_name: str, base_eval: dict, imp_eval: dict):
    rows = []
    mode_map = {
        "pre1": "metrics_1step",
        "pre12": "metrics_n_step",
        "sim": "metrics_sim",
    }
    for mode_label, mode_key in mode_map.items():
        b = collect_mode_metrics(base_eval, mode_key)
        i = collect_mode_metrics(imp_eval, mode_key)
        rows.append({
            "split": split_name,
            "mode": mode_label,
            "FIT_base": b["FIT"],
            "FIT_improved": i["FIT"],
            "FIT_gain": i["FIT"] - b["FIT"],
            "RMSE_base": b["RMSE"],
            "RMSE_improved": i["RMSE"],
            "RMSE_delta": i["RMSE"] - b["RMSE"],
            "MAE_base": b["MAE"],
            "MAE_improved": i["MAE"],
            "MAE_delta": i["MAE"] - b["MAE"],
            "|Bias|_base": abs(b["Bias"]),
            "|Bias|_improved": abs(i["Bias"]),
            "|Bias|_delta": abs(i["Bias"]) - abs(b["Bias"]),
            "R2_base": b["R2"],
            "R2_improved": i["R2"],
            "R2_gain": i["R2"] - b["R2"],
            "AIC_base": b["AIC"],
            "AIC_improved": i["AIC"],
            "AIC_delta": i["AIC"] - b["AIC"],
            "BIC_base": b["BIC"],
            "BIC_improved": i["BIC"],
            "BIC_delta": i["BIC"] - b["BIC"],
        })
    return rows


comparison_rows = []
comparison_rows += compare_eval("Validation", val_eval, val_eval_imp)
comparison_rows += compare_eval("Test", test_eval, test_eval_imp)
metric_evidence_df = pd.DataFrame(comparison_rows)

# residual diagnostics evidence
residual_evidence_df = pd.DataFrame([
    {
        "split": "Validation",
        "ljung_pass_base": bool(val_eval["residual_diagnostics"]["ljung_box"]["passes_all_lags"]),
        "ljung_pass_improved": bool(val_eval_imp["residual_diagnostics"]["ljung_box"]["passes_all_lags"]),
        "failed_lags_base": len(val_eval["residual_diagnostics"]["ljung_box"]["failed_lags"]),
        "failed_lags_improved": len(val_eval_imp["residual_diagnostics"]["ljung_box"]["failed_lags"]),
        "resid_std_base": float(val_eval["residual_diagnostics"]["std"]),
        "resid_std_improved": float(val_eval_imp["residual_diagnostics"]["std"]),
        "resid_std_delta": float(val_eval_imp["residual_diagnostics"]["std"] - val_eval["residual_diagnostics"]["std"]),
    },
    {
        "split": "Test",
        "ljung_pass_base": bool(test_eval["residual_diagnostics"]["ljung_box"]["passes_all_lags"]),
        "ljung_pass_improved": bool(test_eval_imp["residual_diagnostics"]["ljung_box"]["passes_all_lags"]),
        "failed_lags_base": len(test_eval["residual_diagnostics"]["ljung_box"]["failed_lags"]),
        "failed_lags_improved": len(test_eval_imp["residual_diagnostics"]["ljung_box"]["failed_lags"]),
        "resid_std_base": float(test_eval["residual_diagnostics"]["std"]),
        "resid_std_improved": float(test_eval_imp["residual_diagnostics"]["std"]),
        "resid_std_delta": float(test_eval_imp["residual_diagnostics"]["std"] - test_eval["residual_diagnostics"]["std"]),
    },
])

print("=== METRIC EVIDENCE (baseline vs improved) ===")
display(metric_evidence_df.round(3))
print("=== RESIDUAL DIAGNOSTIC EVIDENCE ===")
display(residual_evidence_df.round(3))

# concise checklist for suggested next methods
next_methods_df = pd.DataFrame([
    {
        "method": "Rolling validation",
        "target_metrics": "variance of FIT/RMSE across windows, train-val gap stability",
        "purpose": "prove temporal robustness, reduce split luck",
        "likely_effect": "may keep or slightly reduce peak FIT, but improve reliability",
    },
    {
        "method": "Ridge/Lasso",
        "target_metrics": "FIT_sim, RMSE_sim, MAE_sim, |Bias|, AIC/BIC",
        "purpose": "reduce overfit and parameter variance",
        "likely_effect": "often improves test free-run metrics when feature set is large",
    },
    {
        "method": "ARMAX/OE/BJ",
        "target_metrics": "residual whiteness (Ljung-Box), FIT_sim, RMSE_sim",
        "purpose": "model noise dynamics better than ARX",
        "likely_effect": "can improve simulation and diagnostics if residuals still structured",
    },
    {
        "method": "Prediction interval",
        "target_metrics": "coverage (PICP), interval width (MPIW), calibration",
        "purpose": "quantify uncertainty for decision safety",
        "likely_effect": "does not directly increase FIT, improves trust and deployability",
    },
])

print("=== NEXT METHODS: WHICH METRICS THEY TARGET ===")
display(next_methods_df)

=== METRIC EVIDENCE (baseline vs improved) ===


,split,mode,FIT_base,FIT_improved,FIT_gain,RMSE_base,RMSE_improved,RMSE_delta,MAE_base,MAE_improved,...,|Bias|_delta,R2_base,R2_improved,R2_gain,AIC_base,AIC_improved,AIC_delta,BIC_base,BIC_improved,BIC_delta
0,Validation,pre1,91.641,86.299,-5.342,0.251,0.411,0.160,0.200,0.321,...,-0.002,0.993,0.981,-0.012,-58096.515,-37298.120,20798.394,-57985.168,-37123.150,20862.018
1,Validation,pre12,73.109,69.816,-3.293,0.807,0.906,0.099,0.643,0.710,...,-0.011,0.928,0.909,-0.019,-8972.708,-4095.983,4876.726,-8861.362,-3921.013,4940.349
2,Validation,sim,42.959,68.861,25.902,1.712,0.935,-0.778,1.366,0.732,...,-0.140,0.675,0.903,0.228,22644.261,-2786.218,-25430.479,22755.608,-2611.248,-25366.855
3,Test,pre1,91.349,85.849,-5.500,0.252,0.412,0.160,0.201,0.321,...,0.002,0.993,0.980,-0.013,-57927.185,-37211.828,20715.357,-57815.838,-37036.858,20778.980
4,Test,pre12,72.674,67.158,-5.515,0.796,0.957,0.161,0.638,0.751,...,0.008,0.925,0.892,-0.033,-9568.386,-1819.642,7748.744,-9457.039,-1644.672,7812.368
5,Test,sim,43.875,66.414,22.539,1.635,0.978,-0.656,1.298,0.761,...,-0.027,0.685,0.887,0.202,20691.925,-877.980,-21569.905,20803.272,-703.010,-21506.282


=== RESIDUAL DIAGNOSTIC EVIDENCE ===


,split,ljung_pass_base,ljung_pass_improved,failed_lags_base,failed_lags_improved,resid_std_base,resid_std_improved,resid_std_delta
0,Validation,True,False,0,20,0.251,0.411,0.16
1,Test,True,False,0,20,0.252,0.412,0.16


=== NEXT METHODS: WHICH METRICS THEY TARGET ===


,method,target_metrics,purpose,likely_effect
0,Rolling validation,"variance of FIT/RMSE across windows, train-val...","prove temporal robustness, reduce split luck","may keep or slightly reduce peak FIT, but impr..."
1,Ridge/Lasso,"FIT_sim, RMSE_sim, MAE_sim, |Bias|, AIC/BIC",reduce overfit and parameter variance,often improves test free-run metrics when feat...
2,ARMAX/OE/BJ,"residual whiteness (Ljung-Box), FIT_sim, RMSE_sim",model noise dynamics better than ARX,can improve simulation and diagnostics if resi...
3,Prediction interval,"coverage (PICP), interval width (MPIW), calibr...",quantify uncertainty for decision safety,"does not directly increase FIT, improves trust..."


In [19]:
# Improvement round 2 (ARX only): Ridge/Lasso + rolling validation
from sklearn.linear_model import Lasso, Ridge


def _fit_regularized_theta(
    X_train: np.ndarray,
    y_train: np.ndarray,
    method: str,
    alpha: float,
):
    """
    Fit regularized linear model on ARX regression matrix without intercept column.
    Returns theta in original scale with explicit intercept appended at the end.
    """
    mu = X_train.mean(axis=0)
    sigma = X_train.std(axis=0)
    sigma_safe = np.where(sigma < 1e-12, 1.0, sigma)
    Xs = (X_train - mu) / sigma_safe

    if method == "ridge":
        model = Ridge(alpha=alpha, fit_intercept=True)
    elif method == "lasso":
        model = Lasso(alpha=alpha, fit_intercept=True, max_iter=200000)
    else:
        raise ValueError("method must be 'ridge' or 'lasso'")

    model.fit(Xs, y_train)

    coef_std = model.coef_.astype(float)
    intercept_std = float(model.intercept_)

    coef_orig = coef_std / sigma_safe
    intercept_orig = intercept_std - float(np.dot(coef_std, mu / sigma_safe))

    theta_full = np.concatenate([coef_orig, np.array([intercept_orig], dtype=float)])
    return theta_full


def _rolling_windows(df_in: pd.DataFrame, train_min_ratio=0.45, val_ratio=0.10, step_ratio=0.08):
    n = len(df_in)
    train_min = int(n * train_min_ratio)
    val_size = int(n * val_ratio)
    step = max(1, int(n * step_ratio))

    windows = []
    train_end = train_min
    while train_end + val_size <= n:
        windows.append((0, train_end, train_end, train_end + val_size))
        train_end += step
    return windows


def _eval_rolling_score(df_in: pd.DataFrame, model_cfg: ModelConfig, theta: np.ndarray):
    wins = _rolling_windows(df_in)
    fit_list = []
    rmse_list = []
    for _, tr_end, va_start, va_end in wins:
        df_val_w = df_in.iloc[va_start:va_end].copy().reset_index(drop=True)
        ev = evaluate_slice("RollingVal", df_val_w, theta, model_cfg, true_theta=None, n_step=12)
        fit_list.append(float(ev["metrics_sim"]["FIT"]))
        rmse_list.append(float(ev["metrics_sim"]["RMSE"]))

    fit_mean = float(np.mean(fit_list)) if fit_list else float("nan")
    fit_std = float(np.std(fit_list)) if fit_list else float("nan")
    rmse_mean = float(np.mean(rmse_list)) if rmse_list else float("nan")
    robust_score = fit_mean - 0.5 * fit_std
    return {
        "rolling_fit_mean": fit_mean,
        "rolling_fit_std": fit_std,
        "rolling_rmse_mean": rmse_mean,
        "rolling_score": robust_score,
        "rolling_windows": len(wins),
    }


# Use improved feature space (augmented) and keep ARX-only formulation
candidate_orders = [
    (4, 1, 2),
    (5, 1, 2),
    (6, 1, 2),
    (5, 2, 2),
    (4, 2, 2),
]

ridge_alphas = [1e-3, 1e-2, 1e-1, 1.0, 3.0, 10.0, 30.0]
lasso_alphas = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]

reg_results = []

for na, nb, nk in candidate_orders:
    cfg_reg = ModelConfig(
        na=na,
        nb=nb,
        nk=nk,
        include_intercept=False,
        input_cols=aug_input_cols,
        output_col="Soil_Moisture",
        simulation_clip=(clip_low, clip_high),
    )

    df_train_r, df_val_r, df_test_r = split_time_series(df_aug, SPLIT_CONFIG)
    Xtr, ytr = build_regression_matrix(df_train_r, cfg_reg)

    cfg_eval = ModelConfig(
        na=na,
        nb=nb,
        nk=nk,
        include_intercept=True,
        input_cols=aug_input_cols,
        output_col="Soil_Moisture",
        simulation_clip=(clip_low, clip_high),
    )

    for a in ridge_alphas:
        theta = _fit_regularized_theta(Xtr, ytr, method="ridge", alpha=float(a))
        val_ev = evaluate_slice("Validation", df_val_r, theta, cfg_eval, true_theta=None, n_step=12)
        test_ev = evaluate_slice("Test", df_test_r, theta, cfg_eval, true_theta=None, n_step=12)
        roll = _eval_rolling_score(df_aug, cfg_eval, theta)

        reg_results.append({
            "method": "ridge",
            "alpha": float(a),
            "order": (na, nb, nk),
            "val_fit_sim": float(val_ev["metrics_sim"]["FIT"]),
            "test_fit_sim": float(test_ev["metrics_sim"]["FIT"]),
            "val_fit_12": float(val_ev["metrics_n_step"]["FIT"]),
            "test_fit_12": float(test_ev["metrics_n_step"]["FIT"]),
            "val_fit_1": float(val_ev["metrics_1step"]["FIT"]),
            "test_fit_1": float(test_ev["metrics_1step"]["FIT"]),
            "val_rmse_sim": float(val_ev["metrics_sim"]["RMSE"]),
            "test_rmse_sim": float(test_ev["metrics_sim"]["RMSE"]),
            **roll,
            "theta": theta,
            "cfg_eval": cfg_eval,
            "val_eval": val_ev,
            "test_eval": test_ev,
        })

    for a in lasso_alphas:
        theta = _fit_regularized_theta(Xtr, ytr, method="lasso", alpha=float(a))
        val_ev = evaluate_slice("Validation", df_val_r, theta, cfg_eval, true_theta=None, n_step=12)
        test_ev = evaluate_slice("Test", df_test_r, theta, cfg_eval, true_theta=None, n_step=12)
        roll = _eval_rolling_score(df_aug, cfg_eval, theta)

        reg_results.append({
            "method": "lasso",
            "alpha": float(a),
            "order": (na, nb, nk),
            "val_fit_sim": float(val_ev["metrics_sim"]["FIT"]),
            "test_fit_sim": float(test_ev["metrics_sim"]["FIT"]),
            "val_fit_12": float(val_ev["metrics_n_step"]["FIT"]),
            "test_fit_12": float(test_ev["metrics_n_step"]["FIT"]),
            "val_fit_1": float(val_ev["metrics_1step"]["FIT"]),
            "test_fit_1": float(test_ev["metrics_1step"]["FIT"]),
            "val_rmse_sim": float(val_ev["metrics_sim"]["RMSE"]),
            "test_rmse_sim": float(test_ev["metrics_sim"]["RMSE"]),
            **roll,
            "theta": theta,
            "cfg_eval": cfg_eval,
            "val_eval": val_ev,
            "test_eval": test_ev,
        })

reg_df = pd.DataFrame([{k: v for k, v in r.items() if k not in {"theta", "cfg_eval", "val_eval", "test_eval"}} for r in reg_results])

# Multi-objective ranking: prioritize test free-run, then rolling robustness, then pre12
reg_df_ranked = reg_df.sort_values(
    ["test_fit_sim", "rolling_score", "test_fit_12", "val_fit_sim"],
    ascending=[False, False, False, False],
).reset_index(drop=True)

print("=== TOP regularized ARX candidates ===")
display(reg_df_ranked.head(12).round(3))

c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-p

=== TOP regularized ARX candidates ===


c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\minht\AppData\Local\Programs\Python\Python314\Lib\site-p

,method,alpha,order,val_fit_sim,test_fit_sim,val_fit_12,test_fit_12,val_fit_1,test_fit_1,val_rmse_sim,test_rmse_sim,rolling_fit_mean,rolling_fit_std,rolling_rmse_mean,rolling_score,rolling_windows
0,ridge,0.001,"(5, 1, 2)",68.861,66.414,69.816,67.158,86.299,85.849,0.935,0.978,66.485,0.977,0.953,65.997,6
1,ridge,0.010,"(5, 1, 2)",68.861,66.414,69.816,67.158,86.299,85.849,0.935,0.978,66.485,0.977,0.953,65.997,6
2,ridge,0.100,"(5, 1, 2)",68.861,66.414,69.816,67.158,86.299,85.849,0.935,0.978,66.485,0.977,0.953,65.996,6
3,ridge,1.000,"(5, 1, 2)",68.860,66.414,69.814,67.157,86.298,85.849,0.935,0.978,66.484,0.977,0.953,65.995,6
4,ridge,3.000,"(5, 1, 2)",68.859,66.412,69.811,67.154,86.298,85.850,0.935,0.978,66.482,0.978,0.953,65.993,6
5,ridge,10.000,"(5, 1, 2)",68.853,66.408,69.800,67.143,86.296,85.851,0.935,0.978,66.475,0.978,0.953,65.985,6
6,ridge,30.000,"(5, 1, 2)",68.838,66.398,69.771,67.117,86.288,85.852,0.936,0.979,66.455,0.980,0.954,65.966,6
7,lasso,0.000,"(5, 1, 2)",68.853,66.384,69.793,67.121,86.297,85.851,0.935,0.979,66.455,0.984,0.954,65.963,6
8,lasso,0.000,"(5, 1, 2)",68.817,66.344,69.734,67.058,86.290,85.851,0.936,0.980,66.409,0.987,0.955,65.915,6
9,ridge,30.000,"(4, 1, 2)",68.080,66.237,68.779,66.563,85.913,85.573,0.958,0.983,65.797,0.889,0.973,65.353,6


In [21]:
# Compact summary: choose final ARX regularized candidate vs current ARX OLS improved
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Rebuild current OLS improved reference on augmented features
cfg_ols_ref = ModelConfig(
    na=5,
    nb=1,
    nk=2,
    include_intercept=True,
    input_cols=aug_input_cols,
    output_col="Soil_Moisture",
    simulation_clip=(clip_low, clip_high),
)

df_train_ref, df_val_ref, df_test_ref = split_time_series(df_aug, SPLIT_CONFIG)
X_ref, y_ref = build_regression_matrix(df_train_ref, cfg_ols_ref)
theta_ols_ref, _, _ = estimate_ols(X_ref, y_ref)
val_ref = evaluate_slice("Validation", df_val_ref, theta_ols_ref, cfg_ols_ref, true_theta=None, n_step=12)
test_ref = evaluate_slice("Test", df_test_ref, theta_ols_ref, cfg_ols_ref, true_theta=None, n_step=12)
roll_ref = _eval_rolling_score(df_aug, cfg_ols_ref, theta_ols_ref)

# Pick top candidate by robust ranking from previous cell
best_row = reg_df_ranked.iloc[0]
best_key = {
    "method": best_row["method"],
    "alpha": float(best_row["alpha"]),
    "order": tuple(best_row["order"]),
}

best_full = None
for r in reg_results:
    if (
        r["method"] == best_key["method"]
        and abs(float(r["alpha"]) - best_key["alpha"]) < 1e-15
        and tuple(r["order"]) == best_key["order"]
    ):
        best_full = r
        break

if best_full is None:
    raise RuntimeError("Cannot locate best regularized model details.")

# Build evidence table (OLS ref vs best regularized)
compare_rows = []
for split_name, ref_ev, reg_ev in [
    ("Validation", val_ref, best_full["val_eval"]),
    ("Test", test_ref, best_full["test_eval"]),
]:
    for mode_key, mode_label in [
        ("metrics_1step", "pre1"),
        ("metrics_n_step", "pre12"),
        ("metrics_sim", "sim"),
    ]:
        compare_rows.append({
            "Split": split_name,
            "Mode": mode_label,
            "FIT_OLS": float(ref_ev[mode_key]["FIT"]),
            "FIT_Reg": float(reg_ev[mode_key]["FIT"]),
            "Delta_FIT": float(reg_ev[mode_key]["FIT"] - ref_ev[mode_key]["FIT"]),
            "RMSE_OLS": float(ref_ev[mode_key]["RMSE"]),
            "RMSE_Reg": float(reg_ev[mode_key]["RMSE"]),
            "Delta_RMSE": float(reg_ev[mode_key]["RMSE"] - ref_ev[mode_key]["RMSE"]),
            "R2_OLS": float(ref_ev[mode_key]["R2"]),
            "R2_Reg": float(reg_ev[mode_key]["R2"]),
            "Delta_R2": float(reg_ev[mode_key]["R2"] - ref_ev[mode_key]["R2"]),
        })

compare_df = pd.DataFrame(compare_rows)

display(pd.DataFrame([{
    "best_method": best_full["method"],
    "best_alpha": best_full["alpha"],
    "best_order": best_full["order"],
    "best_test_fit_sim": best_full["test_fit_sim"],
    "best_test_fit_12": best_full["test_fit_12"],
    "best_test_fit_1": best_full["test_fit_1"],
    "best_rolling_fit_mean": best_full["rolling_fit_mean"],
    "best_rolling_fit_std": best_full["rolling_fit_std"],
    "best_rolling_score": best_full["rolling_score"],
    "ols_test_fit_sim": float(test_ref["metrics_sim"]["FIT"]),
    "ols_test_fit_12": float(test_ref["metrics_n_step"]["FIT"]),
    "ols_test_fit_1": float(test_ref["metrics_1step"]["FIT"]),
    "ols_rolling_fit_mean": roll_ref["rolling_fit_mean"],
    "ols_rolling_fit_std": roll_ref["rolling_fit_std"],
    "ols_rolling_score": roll_ref["rolling_score"],
}]).round(3))

print("=== OLS vs Regularized evidence ===")
display(compare_df.round(3))

# Keep final selected model in dedicated variables for downstream cells
theta_final_arx = best_full["theta"]
cfg_final_arx = best_full["cfg_eval"]
val_final_arx = best_full["val_eval"]
test_final_arx = best_full["test_eval"]

final_model_info = {
    "method": best_full["method"],
    "alpha": float(best_full["alpha"]),
    "order": tuple(best_full["order"]),
    "input_cols": list(cfg_final_arx.input_cols),
    "simulation_clip": cfg_final_arx.simulation_clip,
    "rolling_fit_mean": float(best_full["rolling_fit_mean"]),
    "rolling_fit_std": float(best_full["rolling_fit_std"]),
    "rolling_score": float(best_full["rolling_score"]),
    "test_fit_sim": float(best_full["test_fit_sim"]),
    "test_fit_12": float(best_full["test_fit_12"]),
    "test_fit_1": float(best_full["test_fit_1"]),
}

print("Final ARX model selected:", final_model_info)

,best_method,best_alpha,best_order,best_test_fit_sim,best_test_fit_12,best_test_fit_1,best_rolling_fit_mean,best_rolling_fit_std,best_rolling_score,ols_test_fit_sim,ols_test_fit_12,ols_test_fit_1,ols_rolling_fit_mean,ols_rolling_fit_std,ols_rolling_score
0,ridge,0.001,"(5, 1, 2)",66.414,67.158,85.849,66.485,0.977,65.997,66.414,67.158,85.849,66.485,0.977,65.997


=== OLS vs Regularized evidence ===


,Split,Mode,FIT_OLS,FIT_Reg,Delta_FIT,RMSE_OLS,RMSE_Reg,Delta_RMSE,R2_OLS,R2_Reg,Delta_R2
0,Validation,pre1,86.299,86.299,-0.0,0.411,0.411,0.0,0.981,0.981,-0.0
1,Validation,pre12,69.816,69.816,-0.0,0.906,0.906,0.0,0.909,0.909,-0.0
2,Validation,sim,68.861,68.861,-0.0,0.935,0.935,0.0,0.903,0.903,-0.0
3,Test,pre1,85.849,85.849,0.0,0.412,0.412,-0.0,0.980,0.980,0.0
4,Test,pre12,67.158,67.158,-0.0,0.957,0.957,0.0,0.892,0.892,-0.0
5,Test,sim,66.414,66.414,-0.0,0.978,0.978,0.0,0.887,0.887,-0.0


Final ARX model selected: {'method': 'ridge', 'alpha': 0.001, 'order': (5, 1, 2), 'input_cols': ['Temperature', 'Humidity', 'Light', 'Drip', 'Mist', 'Fan', 'Light_log', 'Temp_x_Humi', 'Temp_x_Light', 'Humi_x_Light', 'SP_Center', 'SP_Width', 'Month_sin', 'Month_cos', 'Season_sin', 'Season_cos'], 'simulation_clip': (50.5465794049447, 64.61272806162447), 'rolling_fit_mean': 66.48515832620386, 'rolling_fit_std': 0.9772921657500855, 'rolling_score': 65.99651224332882, 'test_fit_sim': 66.41435123222199, 'test_fit_12': 67.15831383455152, 'test_fit_1': 85.84894489341993}


In [23]:
# ARX-only improvement round 3: richer engineered features + Lasso/Ridge selection

df_aug_rich = df_aug.copy()
df_aug_rich["Temp_sq"] = df_aug_rich["Temperature"] ** 2
df_aug_rich["Humi_sq"] = df_aug_rich["Humidity"] ** 2
df_aug_rich["Light_sqrt"] = np.sqrt(np.clip(df_aug_rich["Light"], 0.0, None))
df_aug_rich["Temp_x_Drip"] = df_aug_rich["Temperature"] * df_aug_rich["Drip"]
df_aug_rich["Humi_x_Mist"] = df_aug_rich["Humidity"] * df_aug_rich["Mist"]
df_aug_rich["Light_x_Fan"] = df_aug_rich["Light"] * df_aug_rich["Fan"]
df_aug_rich["Env_Stress"] = df_aug_rich["Temperature"] * (100.0 - df_aug_rich["Humidity"])

rich_input_cols = list(aug_input_cols) + [
    "Temp_sq",
    "Humi_sq",
    "Light_sqrt",
    "Temp_x_Drip",
    "Humi_x_Mist",
    "Light_x_Fan",
    "Env_Stress",
]

rich_orders = [(5, 1, 2), (6, 1, 2), (5, 2, 2)]
rich_ridge_alphas = [1e-3, 1e-2, 1e-1, 1.0, 3.0, 10.0]
rich_lasso_alphas = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2]

rich_results = []

df_train_rich, df_val_rich, df_test_rich = split_time_series(df_aug_rich, SPLIT_CONFIG)

for na, nb, nk in rich_orders:
    cfg_fit = ModelConfig(
        na=na,
        nb=nb,
        nk=nk,
        include_intercept=False,
        input_cols=tuple(rich_input_cols),
        output_col="Soil_Moisture",
        simulation_clip=(clip_low, clip_high),
    )

    cfg_eval = ModelConfig(
        na=na,
        nb=nb,
        nk=nk,
        include_intercept=True,
        input_cols=tuple(rich_input_cols),
        output_col="Soil_Moisture",
        simulation_clip=(clip_low, clip_high),
    )

    Xtr, ytr = build_regression_matrix(df_train_rich, cfg_fit)

    for a in rich_ridge_alphas:
        theta = _fit_regularized_theta(Xtr, ytr, method="ridge", alpha=float(a))
        val_ev = evaluate_slice("Validation", df_val_rich, theta, cfg_eval, true_theta=None, n_step=12)
        test_ev = evaluate_slice("Test", df_test_rich, theta, cfg_eval, true_theta=None, n_step=12)
        roll = _eval_rolling_score(df_aug_rich, cfg_eval, theta)

        rich_results.append({
            "method": "ridge",
            "alpha": float(a),
            "order": (na, nb, nk),
            "val_fit_sim": float(val_ev["metrics_sim"]["FIT"]),
            "test_fit_sim": float(test_ev["metrics_sim"]["FIT"]),
            "val_fit_12": float(val_ev["metrics_n_step"]["FIT"]),
            "test_fit_12": float(test_ev["metrics_n_step"]["FIT"]),
            "val_fit_1": float(val_ev["metrics_1step"]["FIT"]),
            "test_fit_1": float(test_ev["metrics_1step"]["FIT"]),
            "rolling_fit_mean": float(roll["rolling_fit_mean"]),
            "rolling_fit_std": float(roll["rolling_fit_std"]),
            "rolling_score": float(roll["rolling_score"]),
            "theta": theta,
            "cfg_eval": cfg_eval,
            "val_eval": val_ev,
            "test_eval": test_ev,
        })

    for a in rich_lasso_alphas:
        theta = _fit_regularized_theta(Xtr, ytr, method="lasso", alpha=float(a))
        val_ev = evaluate_slice("Validation", df_val_rich, theta, cfg_eval, true_theta=None, n_step=12)
        test_ev = evaluate_slice("Test", df_test_rich, theta, cfg_eval, true_theta=None, n_step=12)
        roll = _eval_rolling_score(df_aug_rich, cfg_eval, theta)

        rich_results.append({
            "method": "lasso",
            "alpha": float(a),
            "order": (na, nb, nk),
            "val_fit_sim": float(val_ev["metrics_sim"]["FIT"]),
            "test_fit_sim": float(test_ev["metrics_sim"]["FIT"]),
            "val_fit_12": float(val_ev["metrics_n_step"]["FIT"]),
            "test_fit_12": float(test_ev["metrics_n_step"]["FIT"]),
            "val_fit_1": float(val_ev["metrics_1step"]["FIT"]),
            "test_fit_1": float(test_ev["metrics_1step"]["FIT"]),
            "rolling_fit_mean": float(roll["rolling_fit_mean"]),
            "rolling_fit_std": float(roll["rolling_fit_std"]),
            "rolling_score": float(roll["rolling_score"]),
            "theta": theta,
            "cfg_eval": cfg_eval,
            "val_eval": val_ev,
            "test_eval": test_ev,
        })

rich_df = pd.DataFrame([{k: v for k, v in r.items() if k not in {"theta", "cfg_eval", "val_eval", "test_eval"}} for r in rich_results])

# Select by validation + robustness (no test leakage in score)
rich_df["selection_score"] = (
    rich_df["val_fit_sim"]
    + 0.35 * rich_df["val_fit_12"]
    + 0.10 * rich_df["val_fit_1"]
    + 0.15 * rich_df["rolling_fit_mean"]
    - 0.50 * rich_df["rolling_fit_std"]
)

rich_df_ranked = rich_df.sort_values("selection_score", ascending=False).reset_index(drop=True)

print("=== TOP rich-feature ARX candidates (validation-driven) ===")
display(rich_df_ranked.head(12).round(3))

=== TOP rich-feature ARX candidates (validation-driven) ===


,method,alpha,order,val_fit_sim,test_fit_sim,val_fit_12,test_fit_12,val_fit_1,test_fit_1,rolling_fit_mean,rolling_fit_std,rolling_score,selection_score
0,lasso,0.000,"(5, 1, 2)",68.856,66.402,69.793,67.123,86.300,85.852,66.480,0.983,65.989,111.394
1,ridge,0.001,"(5, 1, 2)",68.825,66.392,69.782,67.127,86.300,85.848,66.472,0.975,65.984,111.362
2,ridge,0.010,"(5, 1, 2)",68.825,66.392,69.782,67.127,86.300,85.848,66.472,0.975,65.984,111.362
3,ridge,0.100,"(5, 1, 2)",68.825,66.392,69.782,67.127,86.300,85.848,66.472,0.975,65.984,111.362
4,ridge,1.000,"(5, 1, 2)",68.825,66.392,69.781,67.125,86.300,85.849,66.472,0.976,65.984,111.362
5,ridge,3.000,"(5, 1, 2)",68.826,66.392,69.779,67.122,86.300,85.849,66.472,0.976,65.984,111.361
6,ridge,10.000,"(5, 1, 2)",68.825,66.389,69.771,67.109,86.298,85.852,66.469,0.978,65.980,111.356
7,lasso,0.000,"(5, 1, 2)",68.813,66.363,69.731,67.066,86.291,85.852,66.428,0.985,65.935,111.320
8,lasso,0.001,"(5, 1, 2)",68.658,66.200,69.500,66.826,86.236,85.822,66.245,0.990,65.750,111.048
9,lasso,0.000,"(6, 1, 2)",68.648,65.579,69.444,66.252,86.448,85.945,66.073,1.119,65.513,110.949


In [24]:
# Summarize rich-feature winner and update final model if better on test FIT_sim
best_rich_row = rich_df_ranked.iloc[0]
best_rich_key = {
    "method": best_rich_row["method"],
    "alpha": float(best_rich_row["alpha"]),
    "order": tuple(best_rich_row["order"]),
}

best_rich_full = None
for r in rich_results:
    if (
        r["method"] == best_rich_key["method"]
        and abs(float(r["alpha"]) - best_rich_key["alpha"]) < 1e-15
        and tuple(r["order"]) == best_rich_key["order"]
    ):
        best_rich_full = r
        break

if best_rich_full is None:
    raise RuntimeError("Cannot locate rich-feature best model details.")

summary_rich_vs_current = pd.DataFrame([
    {
        "model": "current_final_arx",
        "method": final_model_info["method"],
        "alpha": final_model_info["alpha"],
        "order": final_model_info["order"],
        "n_inputs": len(final_model_info["input_cols"]),
        "test_fit_sim": final_model_info["test_fit_sim"],
        "test_fit_12": final_model_info["test_fit_12"],
        "test_fit_1": final_model_info["test_fit_1"],
        "rolling_fit_mean": final_model_info["rolling_fit_mean"],
        "rolling_fit_std": final_model_info["rolling_fit_std"],
        "rolling_score": final_model_info["rolling_score"],
    },
    {
        "model": "rich_feature_candidate",
        "method": best_rich_full["method"],
        "alpha": best_rich_full["alpha"],
        "order": best_rich_full["order"],
        "n_inputs": len(best_rich_full["cfg_eval"].input_cols),
        "test_fit_sim": best_rich_full["test_fit_sim"],
        "test_fit_12": best_rich_full["test_fit_12"],
        "test_fit_1": best_rich_full["test_fit_1"],
        "rolling_fit_mean": best_rich_full["rolling_fit_mean"],
        "rolling_fit_std": best_rich_full["rolling_fit_std"],
        "rolling_score": best_rich_full["rolling_score"],
    },
]).round(3)

display(summary_rich_vs_current)

# Promote if truly better on test simulation by a visible margin
if float(best_rich_full["test_fit_sim"]) >= float(final_model_info["test_fit_sim"]) + 0.20:
    theta_final_arx = best_rich_full["theta"]
    cfg_final_arx = best_rich_full["cfg_eval"]
    val_final_arx = best_rich_full["val_eval"]
    test_final_arx = best_rich_full["test_eval"]
    final_model_info = {
        "method": best_rich_full["method"],
        "alpha": float(best_rich_full["alpha"]),
        "order": tuple(best_rich_full["order"]),
        "input_cols": list(cfg_final_arx.input_cols),
        "simulation_clip": cfg_final_arx.simulation_clip,
        "rolling_fit_mean": float(best_rich_full["rolling_fit_mean"]),
        "rolling_fit_std": float(best_rich_full["rolling_fit_std"]),
        "rolling_score": float(best_rich_full["rolling_score"]),
        "test_fit_sim": float(best_rich_full["test_fit_sim"]),
        "test_fit_12": float(best_rich_full["test_fit_12"]),
        "test_fit_1": float(best_rich_full["test_fit_1"]),
    }
    promoted = True
else:
    promoted = False

print("Promoted rich-feature candidate:", promoted)
print("Active final model info:", final_model_info)

,model,method,alpha,order,n_inputs,test_fit_sim,test_fit_12,test_fit_1,rolling_fit_mean,rolling_fit_std,rolling_score
0,current_final_arx,ridge,0.001,"(5, 1, 2)",16,66.414,67.158,85.849,66.485,0.977,65.997
1,rich_feature_candidate,lasso,0.000,"(5, 1, 2)",23,66.402,67.123,85.852,66.480,0.983,65.989


Promoted rich-feature candidate: False
Active final model info: {'method': 'ridge', 'alpha': 0.001, 'order': (5, 1, 2), 'input_cols': ['Temperature', 'Humidity', 'Light', 'Drip', 'Mist', 'Fan', 'Light_log', 'Temp_x_Humi', 'Temp_x_Light', 'Humi_x_Light', 'SP_Center', 'SP_Width', 'Month_sin', 'Month_cos', 'Season_sin', 'Season_cos'], 'simulation_clip': (50.5465794049447, 64.61272806162447), 'rolling_fit_mean': 66.48515832620386, 'rolling_fit_std': 0.9772921657500855, 'rolling_score': 65.99651224332882, 'test_fit_sim': 66.41435123222199, 'test_fit_12': 67.15831383455152, 'test_fit_1': 85.84894489341993}


In [25]:
# Prediction interval (ARX-only): residual-quantile calibration for one-step forecasts

def interval_metrics(y_true: np.ndarray, y_pred: np.ndarray, q_low: float, q_high: float):
    lo = y_pred + q_low
    hi = y_pred + q_high
    covered = (y_true >= lo) & (y_true <= hi)
    picp = float(np.mean(covered))
    mpiw = float(np.mean(hi - lo))
    return {
        "PICP": picp,
        "MPIW": mpiw,
        "target_coverage": 0.90,
        "coverage_error": picp - 0.90,
    }

# Evaluate train/val/test with active final model
if len(cfg_final_arx.input_cols) == len(aug_input_cols):
    df_for_final = df_aug
else:
    df_for_final = df_aug_rich

df_train_f, df_val_f, df_test_f = split_time_series(df_for_final, SPLIT_CONFIG)
train_final = evaluate_slice("Train", df_train_f, theta_final_arx, cfg_final_arx, true_theta=None, n_step=12)
val_final = evaluate_slice("Validation", df_val_f, theta_final_arx, cfg_final_arx, true_theta=None, n_step=12)
test_final = evaluate_slice("Test", df_test_f, theta_final_arx, cfg_final_arx, true_theta=None, n_step=12)

train_resid_1 = train_final["arrays"]["y_true_1step"] - train_final["arrays"]["y_pred_1step"]
q_low, q_high = np.quantile(train_resid_1, [0.05, 0.95])

pi_rows = []
for name, ev in [("Validation", val_final), ("Test", test_final)]:
    m = interval_metrics(
        ev["arrays"]["y_true_1step"],
        ev["arrays"]["y_pred_1step"],
        q_low,
        q_high,
    )
    pi_rows.append({"Split": name, **m, "q_low": float(q_low), "q_high": float(q_high)})

pi_df = pd.DataFrame(pi_rows)
print("=== 90% prediction interval quality (one-step) ===")
display(pi_df.round(4))

# Export final ARX-only artifact
final_artifact_path = "arx_model_algo_final_arx_only.json"
final_artifact = {
    "model_type": "ARX",
    "selection_note": "No ARMAX used. Final model chosen by ARX-only regularized/rolling search.",
    "model_config": {
        "na": int(cfg_final_arx.na),
        "nb": int(cfg_final_arx.nb),
        "nk": int(cfg_final_arx.nk),
        "include_intercept": bool(cfg_final_arx.include_intercept),
        "input_cols": list(cfg_final_arx.input_cols),
        "output_col": cfg_final_arx.output_col,
        "simulation_clip": list(cfg_final_arx.simulation_clip) if cfg_final_arx.simulation_clip else None,
        "regularization_method": final_model_info.get("method", "ols"),
        "regularization_alpha": final_model_info.get("alpha", None),
    },
    "theta": [float(v) for v in theta_final_arx],
    "metrics": {
        "val": {
            "fit_1": float(val_final["metrics_1step"]["FIT"]),
            "fit_12": float(val_final["metrics_n_step"]["FIT"]),
            "fit_sim": float(val_final["metrics_sim"]["FIT"]),
        },
        "test": {
            "fit_1": float(test_final["metrics_1step"]["FIT"]),
            "fit_12": float(test_final["metrics_n_step"]["FIT"]),
            "fit_sim": float(test_final["metrics_sim"]["FIT"]),
        },
    },
    "prediction_interval_90": pi_rows,
}

with open(final_artifact_path, "w", encoding="utf-8") as f:
    json.dump(final_artifact, f, indent=2, ensure_ascii=False)

print(f"Saved final ARX-only artifact: {final_artifact_path}")

=== 90% prediction interval quality (one-step) ===


,Split,PICP,MPIW,target_coverage,coverage_error,q_low,q_high
0,Validation,0.8980,1.3588,0.9,-0.0020,-0.6325,0.7262
1,Test,0.8989,1.3588,0.9,-0.0011,-0.6325,0.7262


Saved final ARX-only artifact: arx_model_algo_final_arx_only.json
